# Análisis de Similitud Curricular — Nivel 1: TF-IDF

**Proyecto:** TIC-D — Similitud curricular entre carreras de ingeniería en Ecuador: un enfoque basado en NLP y taxonomías de competencias  
**Autor:** Glenn  
**Institución:** Escuela Politécnica Nacional (EPN)

---

## ¿Qué es el Nivel 1 y cuál es su rol en la metodología?

El Nivel 1 constituye el **baseline léxico** del análisis. Es el punto de partida más simple y sirve como referencia para evaluar si los niveles más sofisticados (embeddings semánticos y vectores ESCO) producen mejores resultados.

La pregunta que responde este nivel es:

> *¿Qué tan similares son las carreras universitarias cuando comparamos únicamente las palabras que usan para describir sus perfiles?*

### Posición en la metodología

| Nivel | Método | Qué compara | Este notebook |
|---|---|---|---|
| **1** | **TF-IDF** | **Frecuencia de palabras** | **✓ Este notebook** |
| 2 | Embeddings directos | Significado semántico global | `embeddings_analysis.ipynb` |
| 3 | Vectores ESCO | Habilidades estandarizadas | `esco_analysis.ipynb` |

### Limitación principal del Nivel 1

TF-IDF compara **palabras exactas**, no significados. Si dos universidades describen la misma competencia con vocabulario distinto, TF-IDF las verá como diferentes aunque sean curricularmente equivalentes. Esta limitación es precisamente lo que motiva el uso de embeddings semánticos en los niveles siguientes.

---

## Pipeline del Nivel 1

```
PASO 1: Texto de carrera
        perfil_egreso + perfil_profesional
        ↓
PASO 2: Vectorización TF-IDF
        texto → vector disperso de pesos de términos
        (mayor peso = término más representativo de esa carrera)
        ↓
PASO 3: Similitud coseno entre vectores TF-IDF
        sim[i,j] = cos(vector_i, vector_j)
        → matriz 163 × 163
        ↓
PASO 4: Clustering jerárquico + visualizaciones
        dendrograma, heatmap, métricas
```

---

## Estructura del notebook

| Sección | Contenido | Prerequisito |
|---|---|---|
| 0 | Configuración e imports | Siempre |
| 1 | Carga de datos y vectorización TF-IDF | Sección 0 |
| 2 | Dendrograma de similitud léxica | Sección 1 |
| 3 | Heatmap de similitud entre carreras | Sección 1 |
| 4 | Métricas de evaluación | Sección 1 |
| 5 | Análisis de términos más representativos | Sección 1 |
| 6 | Interpretación y conclusiones | Sección 4 |

---
## Sección 0 — Configuración e Imports

**Qué hace esta celda:**  
Importa todas las librerías necesarias y define las rutas del proyecto.

**Librerías y su función específica en este notebook:**

| Librería | Función |
|---|---|
| `pandas` | Cargar y manipular el CSV de carreras |
| `numpy` | Operaciones matriciales sobre vectores TF-IDF |
| `sklearn.feature_extraction.text` | `TfidfVectorizer` — convierte textos en vectores de peso de términos |
| `sklearn.metrics.pairwise` | `cosine_similarity` — calcula similitud entre vectores TF-IDF |
| `sklearn.metrics` | `silhouette_score` — evalúa calidad de los clusters |
| `scipy.cluster.hierarchy` | Clustering jerárquico y dendrograma |
| `scipy.spatial.distance` | `squareform` — convierte matriz de distancias al formato requerido |
| `matplotlib` | Dendrograma, heatmap y tabla de métricas |
| `matplotlib.colors.TwoSlopeNorm` | Normalización dinámica de colores para el heatmap |

**Cuándo ejecutar:** Siempre primero. Si reinicias Jupyter, vuelve a ejecutar esta celda.

In [ ]:
import json
import warnings
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import (
    dendrogram, linkage, cophenet,
    fcluster, leaves_list
)
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

# ── Rutas del proyecto ────────────────────────────────────────────────────────
# Path('..') sube un nivel desde notebooks/ hasta la raíz de TIC-D
BASE_DIR   = Path('..')
PROCESSED  = BASE_DIR / 'data' / 'processed'
OUTPUT_DIR = BASE_DIR / 'outputs' / date.today().isoformat() / 'nivel1'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Paleta de colores por universidad ─────────────────────────────────────────
PALETA = {
    'EPN':'#1a56db','ESPOL':'#0e9f6e','UPS':'#e3a008','ESPOCH':'#9061f9',
    'UCE':'#e02424','UTM':'#ff5a1f','UG':'#057a55','USFQ':'#0694a2',
    'UDLA':'#c81e1e','UTPL':'#5521b5','ESPE':'#1e429f','UCSG':'#723b13',
    'PUCE':'#014737','UTN':'#6b21a8',
}
def get_color(sig):
    return PALETA.get(sig, '#4b5563')

# ── Stopwords en español ──────────────────────────────────────────────────────
# Palabras muy frecuentes que no aportan significado discriminativo.
# Se eliminan antes de construir el vocabulario TF-IDF para que el modelo
# se enfoque en los términos realmente representativos de cada carrera.
STOPWORDS_ES = [
    'de','la','el','en','y','a','los','las','que','se','del','un','una',
    'con','por','para','su','sus','es','son','al','lo','como','más','o',
    'pero','si','le','han','hay','ser','estar','tiene','tienen','puede',
    'pueden','así','este','esta','estos','estas','entre','también','todo',
    'todos','toda','todas','sobre','cuando','donde','cada','mediante',
    'sin','bajo','según','tanto','no','ni','me','te','nos','les','muy',
    'bien','otro','otros','misma','mismos','mismo','hacia','hasta','desde',
]

print('✓ Configuración completada')
print(f'  Directorio base: {BASE_DIR.resolve()}')
print(f'  Salida de hoy:   {OUTPUT_DIR.resolve()}')

---
## Sección 1 — Carga de Datos y Vectorización TF-IDF

**Qué hace esta celda:**  
Carga el dataset de carreras, construye el corpus de texto y lo vectoriza con TF-IDF.

### ¿Qué es TF-IDF?

TF-IDF (Term Frequency — Inverse Document Frequency) es una técnica que convierte un texto en un vector numérico asignando un **peso** a cada término. El peso combina dos factores:

- **TF (frecuencia del término):** qué tan seguido aparece una palabra en ese documento. Palabras que aparecen mucho en un perfil reciben más peso.
- **IDF (frecuencia inversa de documento):** qué tan rara es esa palabra en todos los documentos. Palabras que aparecen en todos los perfiles (como "profesional" o "área") reciben menos peso porque no discriminan entre carreras.

El resultado es que las palabras más **características** de una carrera específica tienen peso alto, mientras que las palabras genéricas tienen peso bajo o nulo.

### Parámetros del vectorizador

| Parámetro | Valor | Por qué |
|---|---|---|
| `max_features` | 3000 | Limita el vocabulario a los 3000 términos más frecuentes — evita ruido por términos rarísimos |
| `ngram_range` | (1,2) | Incluye palabras individuales Y bigramas (pares de palabras consecutivas) — captura expresiones como "trabajo en equipo" o "ingeniería civil" |
| `min_df` | 2 | Un término debe aparecer en al menos 2 documentos — elimina términos únicos sin valor comparativo |
| `sublinear_tf` | True | Usa log(tf) en lugar de tf crudo — suaviza el efecto de palabras que aparecen muchas veces |
| `stop_words` | lista ES | Elimina palabras vacías en español antes de vectorizar |

### El corpus de texto

Por cada carrera concatenamos `perfil_egreso + perfil_profesional`. Estos dos campos son los que mejor describen **qué sabe hacer** un graduado y **dónde puede trabajar** — exactamente lo que queremos comparar entre carreras.

**Parámetro configurable:** `METODO` — método de linkage para el clustering jerárquico

In [ ]:
# ── PARÁMETRO CONFIGURABLE ────────────────────────────────────────────────────
METODO = 'ward'
# Método de linkage para clustering jerárquico.
# 'ward' minimiza la varianza interna de los clusters en cada fusión.
# Es el método más común para datos de texto y produce clusters compactos.
# ─────────────────────────────────────────────────────────────────────────────

# --- Cargar dataset de carreras ---
df_carreras = pd.read_csv(BASE_DIR / 'data' / 'processed' / 'carreras_homologas.csv')
df_carreras['perfil_egreso']      = df_carreras['perfil_egreso'].fillna('')
df_carreras['perfil_profesional'] = df_carreras['perfil_profesional'].fillna('')

# --- Construir corpus de texto ---
# Cada carrera queda representada por la concatenación de sus dos campos clave
corpus = (
    df_carreras['perfil_egreso'] + ' ' + df_carreras['perfil_profesional']
).tolist()

print(f'Dataset cargado:')
print(f'  Carreras:        {len(df_carreras)}')
print(f'  Universidades:   {df_carreras["siglas"].nunique()}')
print(f'  Long. media texto: {np.mean([len(t) for t in corpus]):.0f} chars')
print(f'  Long. máxima:    {max([len(t) for t in corpus])} chars')

# --- Vectorizar con TF-IDF ---
# TfidfVectorizer aplica los siguientes pasos internamente:
#   1. Tokenización: divide el texto en palabras y bigramas
#   2. Eliminación de stopwords
#   3. Cálculo de TF: frecuencia (logarítmica) de cada término en cada documento
#   4. Cálculo de IDF: log(N / df_t) donde N=163 documentos y df_t = documentos con ese término
#   5. TF-IDF = TF × IDF
#   6. Normalización L2: cada vector queda con norma = 1
vectorizer = TfidfVectorizer(
    max_features=3000,
    stop_words=STOPWORDS_ES,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)
tfidf_matrix = vectorizer.fit_transform(corpus)
# tfidf_matrix es una matriz dispersa (sparse): shape (163, vocab_size)
# La mayoría de celdas son 0 porque cada carrera solo contiene
# una fracción del vocabulario total

vocab = vectorizer.get_feature_names_out()
print(f'\nVectorización TF-IDF:')
print(f'  Vocabulario total: {len(vocab)} términos')
print(f'  Matriz shape:      {tfidf_matrix.shape}  (carreras × términos)')
print(f'  Densidad:          {tfidf_matrix.nnz / (tfidf_matrix.shape[0]*tfidf_matrix.shape[1]):.4f}')
print(f'  (densidad baja = matriz dispersa, normal para TF-IDF)')

# --- Calcular similitud coseno entre carreras ---
# cosine_similarity sobre vectores TF-IDF normalizados equivale al producto punto.
# sim[i,j] = 1 → los perfiles de las carreras i y j usan el mismo vocabulario
# sim[i,j] = 0 → los perfiles no comparten ningún término relevante
sim_cc   = cosine_similarity(tfidf_matrix)  # (163, 163)
np.fill_diagonal(sim_cc, 1.0)

# Convertir a distancia para el clustering
dist_cc  = np.clip(1 - sim_cc, 0, None)
np.fill_diagonal(dist_cc, 0)

# Construir árbol de clustering
dist_condensed = squareform(dist_cc, checks=False)
Z = linkage(dist_condensed, method=METODO)

# Estadísticas del rango de similitud
mask     = ~np.eye(sim_cc.shape[0], dtype=bool)
sim_vals = sim_cc[mask]
print(f'\nSimilitud TF-IDF carrera × carrera (sin diagonal):')
print(f'  Mínimo:  {sim_vals.min():.4f}  ← par más distinto léxicamente')
print(f'  Máximo:  {sim_vals.max():.4f}  ← par más similar léxicamente')
print(f'  Media:   {sim_vals.mean():.4f}')
print(f'  Mediana: {np.median(sim_vals):.4f}')
print(f'\n  Nota: rango amplio indica que TF-IDF discrimina bien entre carreras')
print(f'  con vocabularios muy distintos, pero puede fallar cuando distintas')
print(f'  universidades usan sinónimos para las mismas competencias.')

---
## Sección 2 — Dendrograma de Similitud Léxica

**Qué hace esta celda:**  
Construye y visualiza el árbol jerárquico de similitud entre las 163 carreras basado en sus vectores TF-IDF.

### ¿Cómo se construye el dendrograma?

El clustering jerárquico aglomerativo con método Ward funciona así:

1. **Inicio:** cada una de las 163 carreras es su propio cluster
2. **Iteración:** en cada paso, fusiona los dos clusters cuya unión minimice el incremento de varianza interna
3. **Fin:** todas las carreras quedan en un solo cluster

El dendrograma visualiza este proceso de fusiones. Cada hoja es una carrera. Las ramas que se unen cerca del eje Y (distancia pequeña) son carreras muy similares léxicamente. Las que se unen lejos son carreras con vocabulario muy distinto.

### Cómo interpretar el resultado

- **Eje X:** distancia coseno (1 − similitud TF-IDF). Más a la derecha = más distintas
- **Color de etiquetas:** universidad de cada carrera según la paleta
- **Agrupamiento coherente:** carreras del mismo nombre de distintas universidades deberían formar una rama antes de unirse con otras áreas
- **Agrupamiento incoherente:** si carreras de áreas muy distintas quedan juntas, significa que comparten vocabulario genérico pero no son curricularmente similares — esto es la limitación principal de TF-IDF

In [ ]:
# ── PARÁMETRO CONFIGURABLE ────────────────────────────────────────────────────
DPI = 150  # Resolución: 150 para pantalla, 300 para impresión en tesis
# ─────────────────────────────────────────────────────────────────────────────

print(f'Árbol de clustering:')
print(f'  Método:          {METODO}')
print(f'  Altura máxima:   {Z[:,2].max():.4f}')
print(f'  Umbral visual:   {0.6*Z[:,2].max():.4f} (60% de la altura máxima)')

# Etiquetas: "Nombre De La Carrera\nSIGLAS"
etiquetas = (
    df_carreras['nombre'].str.title() + '\n' + df_carreras['siglas']
).tolist()

n     = len(df_carreras)
fig_h = max(22, n * 0.23)
fig, ax = plt.subplots(figsize=(18, fig_h))
fig.patch.set_facecolor('#fafafa')
ax.set_facecolor('#fafafa')

# dendrogram() dibuja el árbol Z como árbol horizontal
# color_threshold: ramas por debajo de este umbral se colorean por cluster
# count_sort='descendent': sub-árboles más grandes arriba
dend = dendrogram(
    Z,
    labels=etiquetas,
    orientation='left',
    ax=ax,
    leaf_font_size=7.5,
    color_threshold=0.6 * Z[:,2].max(),
    above_threshold_color='#9ca3af',
    count_sort='descendent',
)

# Colorear etiquetas por universidad
for tick in ax.get_yticklabels():
    partes = tick.get_text().split('\n')
    if len(partes) >= 2:
        tick.set_color(get_color(partes[-1].strip()))

ax.set_xlabel(
    'Distancia coseno sobre vectores TF-IDF  (1 − similitud léxica)\n'
    '← más similares (mismo vocabulario)              más distintos →',
    fontsize=11, labelpad=10, color='#374151'
)
ax.set_title(
    f'Similitud curricular — Nivel 1: TF-IDF (Baseline léxico)\n'
    f'Vocabulario: {len(vocab)} términos · Bigramas · Linkage: {METODO}\n'
    f'{n} carreras · {df_carreras["siglas"].nunique()} universidades ecuatorianas',
    fontsize=12, fontweight='bold', color='#111827', pad=16,
)
for spine in ['top','right','left']:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis='x', colors='#6b7280', labelsize=9)
ax.grid(axis='x', linestyle='--', alpha=0.4, color='#d1d5db')

# Leyenda de universidades
handles = [
    plt.matplotlib.patches.Patch(
        color=get_color(s),
        label=f'{s}  ({(df_carreras["siglas"]==s).sum()})'
    )
    for s in sorted(df_carreras['siglas'].unique())
]
leg = ax.legend(
    handles=handles, title='Universidad',
    loc='lower right', fontsize=8, title_fontsize=9,
    framealpha=0.9, edgecolor='#e5e7eb', ncol=2
)
leg.get_title().set_color('#374151')
plt.tight_layout(pad=1.5)

fecha    = date.today().isoformat()
png_path = OUTPUT_DIR / f'dendrograma_tfidf_{fecha}.png'
pdf_path = OUTPUT_DIR / f'dendrograma_tfidf_{fecha}.pdf'
fig.savefig(png_path, dpi=DPI, bbox_inches='tight', facecolor='#fafafa')
fig.savefig(pdf_path, bbox_inches='tight', facecolor='#fafafa')
plt.show()
print(f'\n✓ PNG: {png_path}')
print(f'✓ PDF: {pdf_path}')

---
## Sección 3 — Heatmap de Similitud entre Carreras

**Qué hace esta celda:**  
Genera la matriz visual de similitud TF-IDF entre las 163 carreras.

### Diferencia con los heatmaps de los Niveles 2 y 3

| | Nivel 1 (este notebook) | Nivel 2 | Nivel 3 |
|---|---|---|---|
| Qué mide | Similitud de vocabulario | Similitud semántica directa | Similitud de perfil de competencias ESCO |
| Rango típico | Más amplio (0 a ~0.9) | Comprimido (0.7 a 0.99) | Muy comprimido (0.85 a 0.99) |
| Interpretación | ¿Comparten palabras? | ¿Tienen el mismo significado? | ¿Cubren las mismas competencias? |

El rango más amplio del TF-IDF hace que el heatmap sea más fácil de leer visualmente — hay mayor contraste entre similares y distintos. Sin embargo, eso no significa que sea mejor: simplemente penaliza más las diferencias de vocabulario.

### Por qué se usa TwoSlopeNorm

Aunque TF-IDF tiene un rango más amplio que los otros niveles, igual es conveniente usar rango dinámico para centrar el colormap en el punto medio real de los valores, garantizando que el amarillo del colormap corresponda exactamente a la similitud media observada.

In [ ]:
# ── PARÁMETRO CONFIGURABLE ────────────────────────────────────────────────────
DPI = 150
# ─────────────────────────────────────────────────────────────────────────────

# Reordenar según clustering para que similares queden adyacentes
orden        = leaves_list(Z)
sim_ordenada = sim_cc[np.ix_(orden, orden)]

# Etiquetas cortas ordenadas
etiquetas_h = [
    f"{df_carreras['nombre'].iloc[i][:28]}"
    f"{'...' if len(df_carreras['nombre'].iloc[i]) > 28 else ''}"
    f" — {df_carreras['siglas'].iloc[i]}"
    for i in orden
]

# Rango dinámico
mask_d  = ~np.eye(sim_ordenada.shape[0], dtype=bool)
vmin_r  = sim_ordenada[mask_d].min()
vmax_r  = sim_ordenada[mask_d].max()
vcenter = (vmin_r + vmax_r) / 2
norm    = TwoSlopeNorm(vmin=vmin_r, vcenter=vcenter, vmax=vmax_r)

print(f'Rango de similitud TF-IDF (sin diagonal):')
print(f'  Mínimo:  {vmin_r:.4f}  ← par más distinto léxicamente')
print(f'  Máximo:  {vmax_r:.4f}  ← par más similar (sin ser la misma carrera)')
print(f'  Centro:  {vcenter:.4f}  ← punto medio del colormap')
print(f'  Rango:   {vmax_r-vmin_r:.4f}  ← TF-IDF tiene rango más amplio que embeddings')

n        = len(df_carreras)
fig_size = max(20, n * 0.18)
fig, ax  = plt.subplots(figsize=(fig_size, fig_size))
fig.patch.set_facecolor('#fafafa')

im = ax.imshow(
    sim_ordenada,
    cmap='RdYlGn',
    norm=norm,
    aspect='auto',
    interpolation='nearest',
)
plt.colorbar(
    im, ax=ax,
    label=f'Similitud coseno TF-IDF (rango: {vmin_r:.3f} – {vmax_r:.3f})',
    shrink=0.6, pad=0.02
)

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(etiquetas_h, rotation=90, fontsize=5.5, ha='right')
ax.set_yticklabels(etiquetas_h, fontsize=5.5)

for i, tick in enumerate(ax.get_yticklabels()):
    tick.set_color(get_color(df_carreras['siglas'].iloc[orden[i]]))
for i, tick in enumerate(ax.get_xticklabels()):
    tick.set_color(get_color(df_carreras['siglas'].iloc[orden[i]]))

ax.set_title(
    f'Matriz de similitud — Nivel 1: TF-IDF (Baseline léxico)\n'
    f'Rango dinámico (TwoSlopeNorm) · Ordenado por clustering {METODO}\n'
    f'{n} carreras · {df_carreras["siglas"].nunique()} universidades',
    fontsize=11, fontweight='bold', color='#111827', pad=14,
)
plt.tight_layout(pad=1.5)

fecha    = date.today().isoformat()
png_path = OUTPUT_DIR / f'heatmap_tfidf_{fecha}.png'
pdf_path = OUTPUT_DIR / f'heatmap_tfidf_{fecha}.pdf'
fig.savefig(png_path, dpi=DPI, bbox_inches='tight', facecolor='#fafafa')
fig.savefig(pdf_path, bbox_inches='tight', facecolor='#fafafa')
plt.show()
print(f'\n✓ PNG: {png_path}')
print(f'✓ PDF: {pdf_path}')

---
## Sección 4 — Métricas de Evaluación

**Qué hace esta celda:**  
Calcula las tres métricas de evaluación del clustering y las guarda como JSON para uso posterior en la comparación entre niveles.

### Las tres métricas

**Cophenetic Correlation**  
Mide qué tan fielmente el dendrograma preserva las distancias TF-IDF originales entre carreras.  
Rango [0,1] — más alto es mejor. > 0.7: Bueno | 0.6–0.7: Aceptable | < 0.6: Débil

**Silhouette Score**  
Para cada carrera: ¿qué tan parecida es a las del mismo cluster vs las del cluster vecino?  
Usamos `metric='precomputed'` con la matriz de distancias ya calculada.  
Rango [-1,1] — más alto es mejor. > 0.5: bien definidos | 0.25–0.5: estructura débil

**Pureza de clusters**  
Con k=10: qué porcentaje de carreras en cada cluster comparte el nombre más frecuente.  
Métrica estricta — subestima la calidad real porque no reconoce nombres equivalentes.

### Por qué la pureza del TF-IDF puede ser alta

TF-IDF puede tener pureza alta porque carreras con el mismo nombre en distintas universidades suelen usar vocabulario muy similar (copian o adaptan los mismos términos técnicos). Sin embargo eso no significa que el modelo sea mejor — simplemente agrupa por similitud léxica superficial.

In [ ]:
fecha = date.today().isoformat()
print(f'Métricas de evaluación — Nivel 1: TF-IDF | Método: {METODO}')
print('=' * 60)

# ── 1. Cophenetic Correlation ──────────────────────────────────────────────────
cophenetic_corr, _ = cophenet(Z, dist_condensed)
interp_coph = (
    'Excelente (>0.8)'    if cophenetic_corr > 0.8 else
    'Bueno (0.7-0.8)'     if cophenetic_corr > 0.7 else
    'Aceptable (0.6-0.7)' if cophenetic_corr > 0.6 else
    'Débil (<0.6)'
)
print(f'\n1. Cophenetic Correlation')
print(f'   Valor:          {cophenetic_corr:.4f}')
print(f'   Interpretación: {interp_coph}')
print(f'   Significado:    el dendrograma preserva el {cophenetic_corr*100:.1f}%')
print(f'                   de la estructura de distancias TF-IDF originales')

# ── 2. Silhouette Score ────────────────────────────────────────────────────────
print(f'\n2. Silhouette Score (metric=precomputed sobre dist_cc)')
sil_scores = {}
for k in [5, 7, 10, 15]:
    labels   = fcluster(Z, k, criterion='maxclust')
    score    = silhouette_score(dist_cc, labels, metric='precomputed')
    sil_scores[k] = score
    interp   = (
        'Clusters bien definidos' if score > 0.5  else
        'Estructura moderada'     if score > 0.25 else
        'Sin estructura clara'
    )
    print(f'   k={k:2d}: {score:+.4f}  →  {interp}')

k_opt = max(sil_scores, key=sil_scores.get)
print(f'   Mejor k: {k_opt} (score: {sil_scores[k_opt]:.4f})')

# ── 3. Pureza de clusters ──────────────────────────────────────────────────────
print(f'\n3. Pureza de Clusters (k=10)')
labels_10 = fcluster(Z, 10, criterion='maxclust')
df_ev     = df_carreras[['siglas','nombre']].copy()
df_ev['cluster'] = labels_10
df_ev['nn']      = df_ev['nombre'].str.lower().str.strip()

pureza_clusters = {}
for cid in sorted(df_ev['cluster'].unique()):
    g  = df_ev[df_ev['cluster']==cid]
    vc = g['nn'].value_counts()
    p  = vc.iloc[0] / len(g)
    pureza_clusters[cid] = {'n': len(g), 'mayoria': vc.index[0], 'pureza': p}

pureza_prom = sum(v['pureza'] for v in pureza_clusters.values()) / len(pureza_clusters)

for cid, info in pureza_clusters.items():
    barra = '█' * int(info['pureza'] * 20)
    print(f'   Cluster {cid:2d}: {info["pureza"]:.2f} {barra}'
          f'  ({info["n"]} carreras | mayoría: {info["mayoria"][:40]})')

interp_pur = (
    'Alta (>0.8)'     if pureza_prom > 0.8 else
    'Media (0.6-0.8)' if pureza_prom > 0.6 else
    'Baja (<0.6)'
)
print(f'\n   Pureza promedio: {pureza_prom:.4f}  →  {interp_pur}')
print(f'   Nota: pureza alta en TF-IDF puede reflejar similitud léxica superficial,')
print(f'         no necesariamente equivalencia curricular real.')

# ── Guardar métricas como JSON ─────────────────────────────────────────────────
metricas_n1 = {
    'nivel':                    1,
    'modelo':                   'TF-IDF',
    'parametros': {
        'max_features':         3000,
        'ngram_range':          '(1,2)',
        'min_df':               2,
        'sublinear_tf':         True,
        'vocabulario_final':    int(len(vocab)),
    },
    'metodo_linkage':           METODO,
    'n_carreras':               len(df_carreras),
    'cophenetic':               round(float(cophenetic_corr), 4),
    'interpretacion_cophenetic':interp_coph,
    'silhouette':               {f'k={k}': round(float(v), 4) for k,v in sil_scores.items()},
    'silhouette_k_optimo':      k_opt,
    'pureza_k10':               round(float(pureza_prom), 4),
    'interpretacion_pureza':    interp_pur,
    'pureza_por_cluster': {
        str(k): {
            'n':      v['n'],
            'pureza': round(v['pureza'], 4),
            'nombre_mayoria': v['mayoria']
        }
        for k, v in pureza_clusters.items()
    },
    'rango_similitud': {
        'min': round(float(sim_vals.min()), 4),
        'max': round(float(sim_vals.max()), 4),
        'media': round(float(sim_vals.mean()), 4),
    },
    'fecha': fecha,
}

json_path = OUTPUT_DIR / f'metricas_nivel1_tfidf_{fecha}.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(metricas_n1, f, ensure_ascii=False, indent=2)
print(f'\n✓ Métricas guardadas: {json_path}')

---
## Sección 5 — Términos más Representativos por Área

**Qué hace esta celda:**  
Identifica los términos TF-IDF más importantes por cluster, es decir, las palabras que mejor caracterizan cada grupo de carreras según el modelo léxico.

**Por qué es útil:**  
Esta sección es exclusiva del Nivel 1 — los embeddings no tienen esta capacidad de interpretación directa porque sus dimensiones no tienen nombre. Con TF-IDF, cada dimensión del vector corresponde a un término concreto, lo que permite identificar qué vocabulario distingue a cada cluster.

**Cómo interpretarlo:**  
Los términos con mayor peso promedio en un cluster son los que mejor caracterizan léxicamente a ese grupo de carreras. Si el cluster agrupa carreras coherentes, los términos deberían reflejar el área de conocimiento. Si los términos son genéricos, el cluster mezcla carreras que comparten vocabulario pero no área.

In [ ]:
# ── PARÁMETRO CONFIGURABLE ────────────────────────────────────────────────────
TOP_TERMS = 10  # Número de términos más representativos por cluster
K_CLUSTERS = 10  # Número de clusters a analizar
# ─────────────────────────────────────────────────────────────────────────────

labels_k = fcluster(Z, K_CLUSTERS, criterion='maxclust')
tfidf_arr = tfidf_matrix.toarray()  # convertir sparse a dense para indexar

print(f'Top-{TOP_TERMS} términos TF-IDF más representativos por cluster (k={K_CLUSTERS})')
print('=' * 70)

resultados_terminos = {}
for cid in sorted(set(labels_k)):
    # Índices de carreras en este cluster
    idx_cluster = np.where(labels_k == cid)[0]
    carreras_cluster = df_carreras['nombre'].iloc[idx_cluster].tolist()

    # Peso promedio de cada término en las carreras del cluster
    pesos_promedio = tfidf_arr[idx_cluster].mean(axis=0)

    # Top términos
    top_idx   = np.argsort(pesos_promedio)[::-1][:TOP_TERMS]
    top_terms = [(vocab[i], round(float(pesos_promedio[i]), 4)) for i in top_idx]

    resultados_terminos[cid] = {
        'n_carreras': len(idx_cluster),
        'carreras': carreras_cluster,
        'top_terminos': top_terms,
    }

    print(f'\nCluster {cid:2d} ({len(idx_cluster)} carreras):')
    # Mostrar nombres únicos de carrera en el cluster
    nombres_unicos = list(dict.fromkeys(
        [n.lower() for n in carreras_cluster]
    ))[:5]
    print(f'  Carreras: {", ".join(nombres_unicos[:3])}{" ..." if len(nombres_unicos)>3 else ""}')
    print(f'  Términos más representativos:')
    for term, peso in top_terms:
        barra = '█' * int(peso * 200)
        print(f'    {peso:.4f}  {barra}  {term}')

# Guardar resultados
json_terms = OUTPUT_DIR / f'terminos_por_cluster_tfidf_{date.today().isoformat()}.json'
with open(json_terms, 'w', encoding='utf-8') as f:
    json.dump(
        {str(k): v for k,v in resultados_terminos.items()},
        f, ensure_ascii=False, indent=2
    )
print(f'\n✓ Términos por cluster guardados: {json_terms}')

---
## Sección 6 — Interpretación y Conclusiones del Nivel 1

**Qué hace esta celda:**  
Genera el resumen de hallazgos del Nivel 1 y la narrativa que conecta con los niveles siguientes.

**Cuándo ejecutar:**  
Después de la Sección 4. El texto generado es un punto de partida — edítalo con los valores reales obtenidos y tu propia interpretación.

In [ ]:
print('=' * 70)
print('RESUMEN DE HALLAZGOS — Nivel 1: TF-IDF (Baseline léxico)')
print('=' * 70)
print(f"""
DATASET
  · {len(df_carreras)} carreras de {df_carreras['siglas'].nunique()} universidades ecuatorianas
  · Vocabulario TF-IDF: {len(vocab)} términos (unigramas + bigramas)
  · Campos analizados: perfil_egreso + perfil_profesional

MÉTRICAS OBTENIDAS
  · Cophenetic correlation: {cophenetic_corr:.4f}  →  {interp_coph}
  · Silhouette k=10:        {sil_scores[10]:.4f}
  · Pureza k=10:            {pureza_prom:.4f}  →  {interp_pur}

QUÉ HACE BIEN EL TF-IDF
  · Agrupa carreras que usan vocabulario técnico similar
  · Es simple, rápido y completamente interpretable
  · La pureza puede ser alta cuando carreras del mismo nombre
    usan los mismos términos técnicos entre universidades

LIMITACIÓN PRINCIPAL
  · No captura significado: dos frases con el mismo sentido
    pero vocabulario distinto aparecen como poco similares
  · Ejemplo: "diseña sistemas computacionales" (EPN) vs
    "desarrolla soluciones informáticas" (ESPOL) → similitud baja
    en TF-IDF aunque describen lo mismo
  · Esta limitación motiva el uso de embeddings semánticos
    en los Niveles 2 y 3

NARRATIVA DE TRANSICIÓN HACIA EL NIVEL 2
  El Nivel 1 establece el baseline léxico del análisis. Sus métricas
  sirven como referencia para evaluar si los modelos semánticos
  (BGE-M3 y E5-Large en el Nivel 2) producen agrupamientos más
  coherentes al capturar significado más allá de palabras exactas.
""")
print('=' * 70)